## Import Modules and Dataset

In [1]:
import pandas as pd
import data_analysis_utils as utils
import selectKclassification as k_class

inp_dataset = pd.read_csv('kidney_disease.csv')
inp_dataset = pd.read_csv('kidney_disease.csv').drop(columns='id')

inp_dataset


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35,7300,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,140.0,...,47,6700,4.9,no,no,no,good,no,no,notckd
396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,75.0,...,54,7800,6.2,no,no,no,good,no,no,notckd
397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,100.0,...,49,6600,5.4,no,no,no,good,no,no,notckd
398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,114.0,...,51,7200,5.9,no,no,no,good,no,no,notckd


## Analysing Raw data and Preprocess

In [2]:
print('Missing Value Percentages:', inp_dataset.isnull().mean()*100)

quan, qual = utils.Preprocessing.quanQual(inp_dataset)
inp_dataset = utils.Preprocessing.quanVariables(inp_dataset, qual)


Missing Value Percentages: age                2.25
bp                 3.00
sg                11.75
al                11.50
su                12.25
rbc               38.00
pc                16.25
pcc                1.00
ba                 1.00
bgr               11.00
bu                 4.75
sc                 4.25
sod               21.75
pot               22.00
hemo              13.00
pcv               17.50
wc                26.25
rc                32.50
htn                0.50
dm                 0.50
cad                0.50
appet              0.25
pe                 0.25
ane                0.25
classification     0.00
dtype: float64


In [3]:
inp_dataset[['pcv', 'rc', 'wc']]=inp_dataset[['pcv', 'rc', 'wc']].astype(float)
quan, qual = utils.Preprocessing.quanQual(inp_dataset)


In [4]:
inp_dataset = utils.Preprocessing.simple_missing(inp_dataset, quan, qual)
inp_dataset = utils.Preprocessing.model_missing(inp_dataset, quan)


## Input, Target variables split

In [5]:
data = pd.get_dummies(inp_dataset, drop_first=True)
indep_X = data.iloc[:, :-1]
dep_Y = data.iloc[:, -1]


## Feature Selection

In [6]:
df_results = k_class.selectk_Classification(indep_X, dep_Y, k_no=5)
print(df_results)


           K_No  Logistic      SVMl     SVMnl       KNN     Navie  Decision  \
ChiSquare     5   0.94382  0.955056  0.977528  0.910112  0.865169  0.955056   

             Random     Selected_Features  
ChiSquare  0.977528  bgr, bu, sc, pcv, wc  


In [7]:
fullresult = pd.DataFrame()
for n in range(3,7):
    result = k_class.selectk_Classification(indep_X, dep_Y, n)
    fullresult = pd.concat([fullresult, result], ignore_index=True)


fullresult


,K_No,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random,Selected_Features
0,3,0.764045,0.786517,0.820225,0.797753,0.831461,0.887640,0.842697,"bgr, bu, wc"
1,4,0.831461,0.865169,0.865169,0.831461,0.842697,0.932584,0.943820,"bgr, bu, sc, wc"
2,5,0.943820,0.955056,0.977528,0.910112,0.865169,0.955056,0.977528,"bgr, bu, sc, pcv, wc"
3,6,0.966292,0.966292,0.988764,0.955056,0.943820,0.943820,0.977528,"al, bgr, bu, sc, pcv, wc"


In [13]:
max_val = fullresult.iloc[:, 1:-1].values.max()
row, col = fullresult.iloc[:, 1:-1].stack().idxmax()
k_val = fullresult.iloc[row, 0]
features = fullresult.iloc[row,-1]
print(f'Max Accuracy is: {max_val:.4f}% for {col} Model \nWith K = {k_val}, Selected Features: {features}')


Max Accuracy is: 0.9888% for SVMnl Model 
With K = 6, Selected Features: al, bgr, bu, sc, pcv, wc
